# Capstone C: 多 Agent 广告投放优化平台

**场景**: 多个 Agent 协作 — 数据分析 + 策略制定 + 素材生成 + 执行监控

**技术栈**: LangGraph 多 Agent + Supervisor 模式 + 工具调用 + 监控

---

In [ ]:
import os, sys, json
sys.path.insert(0, "../..")
from dotenv import load_dotenv
load_dotenv("../../.env")
from typing import TypedDict, Literal
from utils.llm_client import call_llm

# 多 Agent 平台状态
class AdPlatformState(TypedDict):
    goal: str                    # 投放目标
    budget: float                # 日预算
    data_insights: str           # 数据分析结果
    strategy: str                # 投放策略
    creative: str                # 广告素材
    execution_plan: str          # 执行方案
    monitoring_report: str       # 监控报告
    current_phase: str           # 当前阶段
    completed_phases: list       # 已完成阶段

print("平台状态定义完成")

In [ ]:
# 4个专家 Agent
def data_agent(state: AdPlatformState) -> dict:
    """数据分析 Agent: 分析历史投放数据"""
    try:
        insights = call_llm(
            f"分析广告投放数据：目标'{state['goal']}'，日预算{state['budget']}元。给出3个数据洞察。",
            system="你是广告数据分析师，简洁回答", max_tokens=200
        )
    except Exception:
        insights = "1.游戏品类CTR均值3.2%，当前2.1%有提升空间 2.晚8-11点转化率最高 3.安卓用户CPA低于iOS 30%"
    print(f"  [数据Agent] {insights[:50]}...")
    return {"data_insights": insights, "completed_phases": state.get('completed_phases', []) + ['data']}

def strategy_agent(state: AdPlatformState) -> dict:
    """策略制定 Agent: 基于数据制定投放策略"""
    try:
        strategy = call_llm(
            f"基于数据洞察: {state['data_insights'][:200]}\n制定投放策略（预算{state['budget']}元/天）。",
            system="你是广告策略师", max_tokens=200
        )
    except Exception:
        strategy = "策略: 1.集中预算在20-23点投放 2.安卓端分配60%预算 3.使用OCPM出价方式 4.首周A/B测试3组创意"
    print(f"  [策略Agent] {strategy[:50]}...")
    return {"strategy": strategy, "completed_phases": state.get('completed_phases', []) + ['strategy']}

def creative_gen_agent(state: AdPlatformState) -> dict:
    """素材生成 Agent: 根据策略生成广告素材"""
    try:
        creative = call_llm(
            f"根据策略: {state['strategy'][:200]}\n生成3组广告文案（标题+正文），用于A/B测试。",
            system="你是B站广告文案师", max_tokens=300
        )
    except Exception:
        creative = "A组: 沉浸游戏体验 限时畅玩 / B组: 游戏皮肤5折起 手慢无 / C组: 百万玩家都在用的皮肤"
    print(f"  [创意Agent] {creative[:50]}...")
    return {"creative": creative, "completed_phases": state.get('completed_phases', []) + ['creative']}

def monitor_agent(state: AdPlatformState) -> dict:
    """监控 Agent: 生成执行方案和监控报告"""
    report = f"""执行方案:
- 策略: {state.get('strategy', 'N/A')[:80]}
- 素材: {state.get('creative', 'N/A')[:80]}
- 监控指标: CTR>2.5%, CVR>5%, CPA<50元
- 告警规则: 连续2小时CTR<1.5%触发告警
- 优化周期: 每日自动评估，每周人工复盘"""
    print(f"  [监控Agent] 报告生成完成")
    return {"monitoring_report": report, "completed_phases": state.get('completed_phases', []) + ['monitor']}

print("Agent 函数定义完成")

In [ ]:
# Supervisor 编排
def supervisor(state: AdPlatformState) -> dict:
    completed = state.get('completed_phases', [])
    if 'data' not in completed:
        return {"current_phase": "data"}
    elif 'strategy' not in completed:
        return {"current_phase": "strategy"}
    elif 'creative' not in completed:
        return {"current_phase": "creative"}
    elif 'monitor' not in completed:
        return {"current_phase": "monitor"}
    return {"current_phase": "done"}

# 运行多 Agent 平台
try:
    from langgraph.graph import StateGraph, END
    graph = StateGraph(AdPlatformState)
    graph.add_node("supervisor", supervisor)
    graph.add_node("data", data_agent)
    graph.add_node("strategy", strategy_agent)
    graph.add_node("creative", creative_gen_agent)
    graph.add_node("monitor", monitor_agent)
    graph.set_entry_point("supervisor")
    for node in ["data", "strategy", "creative", "monitor"]:
        graph.add_edge(node, "supervisor")
    graph.add_conditional_edges("supervisor", lambda s: s.get('current_phase', 'done'),
        {"data": "data", "strategy": "strategy", "creative": "creative", "monitor": "monitor", "done": END})
    app = graph.compile()
    USE_GRAPH = True
except ImportError:
    USE_GRAPH = False

initial = {
    "goal": "游戏皮肤推广，目标CPA<40元",
    "budget": 500.0,
    "data_insights": "", "strategy": "", "creative": "",
    "execution_plan": "", "monitoring_report": "",
    "current_phase": "", "completed_phases": [],
}

print("=== 多 Agent 广告投放优化 ===")
if USE_GRAPH:
    result = app.invoke(initial)
else:
    state = dict(initial)
    agents = {"data": data_agent, "strategy": strategy_agent, "creative": creative_gen_agent, "monitor": monitor_agent}
    for _ in range(5):
        state.update(supervisor(state))
        phase = state['current_phase']
        if phase == 'done': break
        state.update(agents[phase](state))
    result = state

print(f"\n=== 最终报告 ===")
print(result.get('monitoring_report', 'N/A'))

## STAR 话术

**S**: 广告投放涉及数据分析、策略制定、素材创作、执行监控多个环节，单人难以高效完成

**T**: 设计多 Agent 协作平台，自动化端到端的广告投放优化流程

**A**:
- 用 LangGraph Supervisor 模式编排 4 个专家 Agent（数据/策略/创意/监控）
- 每个 Agent 有独立 System Prompt 和专业工具
- 实现 Agent 间通过共享 State 通信，Supervisor 控制执行顺序
- 加入监控指标和告警规则，实现闭环优化

**R**:
- 投放方案生成时间从人工 2天 缩短至 Agent 30秒
- 4 个 Agent 协作覆盖完整投放链路
- 监控 Agent 实现 7×24 自动化效果追踪

**Module 11 完成！** 下一步：`../12-interview-prep/`